# Task 2 — Model Building and Training
Trains Logistic Regression, XGBoost, and LightGBM on both datasets. Evaluates with AUC-PR, F1, and Confusion Matrix.

In [ ]:
import sys, warnings
sys.path.append('..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay
import joblib, os

from src.data_loader import load_fraud_data, load_ip_country, load_creditcard_data
from src.preprocessor import (preprocess_fraud_data, preprocess_creditcard_data,
                               apply_smote, apply_undersampling)
from src.trainer import get_models, evaluate, cross_val_evaluate, save_model

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)
print("Imports OK")


## Part A — Fraud_Data.csv

### A1. Load & Preprocess

In [ ]:
fraud_raw = load_fraud_data('../data/raw/Fraud_Data.csv')
ip_country = load_ip_country('../data/raw/IpAddress_to_Country.csv')

fraud_df, scaler_fraud = preprocess_fraud_data(fraud_raw, ip_country)
print("Preprocessed shape:", fraud_df.shape)
fraud_df.head(2)


### A2. Train-Test Split

In [ ]:
TARGET = 'class'
X_fraud = fraud_df.drop(columns=[TARGET])
y_fraud = fraud_df[TARGET]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud, y_fraud, test_size=0.2, stratify=y_fraud, random_state=42)

print("Train:", X_train_f.shape, "| Fraud rate:", y_train_f.mean().round(4))
print("Test :", X_test_f.shape,  "| Fraud rate:", y_test_f.mean().round(4))


### A3. Apply SMOTE (training set only)

In [ ]:
X_train_f_res, y_train_f_res = apply_smote(X_train_f, y_train_f)
print("After SMOTE:", y_train_f_res.value_counts().to_dict())


### A4. Train & Evaluate All Models

In [ ]:
models_fraud = get_models(dataset='fraud')
results_fraud = {}

for name, model in models_fraud.items():
    print(f"Training {name}...")
    model.fit(X_train_f_res, y_train_f_res)
    metrics = evaluate(model, X_test_f, y_test_f)
    results_fraud[name] = metrics
    save_model(model, name, 'fraud')
    print(f"  F1={metrics['f1']:.4f}  AUC-PR={metrics['auc_pr']:.4f}  ROC-AUC={metrics['roc_auc']:.4f}")

print("\nDone.")


### A5. Cross-Validation (k=5)

In [ ]:
cv_results_fraud = {}
for name, model in models_fraud.items():
    print(f"CV for {name}...")
    cv = cross_val_evaluate(model, X_fraud, y_fraud, k=5)
    cv_results_fraud[name] = cv
    print(f"  CV AUC-PR: {cv['cv_auc_pr_mean']:.4f} ± {cv['cv_auc_pr_std']:.4f}")
    print(f"  CV F1    : {cv['cv_f1_mean']:.4f} ± {cv['cv_f1_std']:.4f}")


### A6. Model Comparison Table

In [ ]:
rows = []
for name in models_fraud:
    m = results_fraud[name]
    cv = cv_results_fraud[name]
    rows.append({
        "Model": name,
        "F1": round(m['f1'], 4),
        "Precision": round(m['precision'], 4),
        "Recall": round(m['recall'], 4),
        "AUC-PR": round(m['auc_pr'], 4),
        "ROC-AUC": round(m['roc_auc'], 4),
        "CV AUC-PR": f"{cv['cv_auc_pr_mean']:.4f} ± {cv['cv_auc_pr_std']:.4f}",
        "CV F1": f"{cv['cv_f1_mean']:.4f} ± {cv['cv_f1_std']:.4f}",
    })

comparison_fraud = pd.DataFrame(rows).set_index("Model")
print("=== Fraud_Data.csv Model Comparison ===")
comparison_fraud


### A7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(models_fraud), figsize=(5*len(models_fraud), 4))
for ax, (name, metrics) in zip(axes, results_fraud.items()):
    disp = ConfusionMatrixDisplay(metrics['confusion_matrix'],
                                   display_labels=['Legit','Fraud'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')
plt.suptitle('Confusion Matrices — Fraud_Data.csv', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrices_fraud.png', bbox_inches='tight')
plt.show()


### A8. Best Model Selection — Fraud_Data

In [ ]:
# Select based on AUC-PR (best metric for imbalanced data)
best_fraud_name = comparison_fraud['AUC-PR'].idxmax()
best_fraud_model = models_fraud[best_fraud_name]
print(f"Best model for Fraud_Data: {best_fraud_name}")
print(results_fraud[best_fraud_name]['report'])


---
## Part B — creditcard.csv

### B1. Load & Preprocess

In [ ]:
cc_raw = load_creditcard_data('../data/raw/creditcard.csv')
cc_df, scaler_cc = preprocess_creditcard_data(cc_raw)
print("Preprocessed shape:", cc_df.shape)
cc_df['Class'].value_counts()


### B2. Train-Test Split

In [ ]:
X_cc = cc_df.drop(columns=['Class'])
y_cc = cc_df['Class']

X_train_cc, X_test_cc, y_train_cc, y_test_cc = train_test_split(
    X_cc, y_cc, test_size=0.2, stratify=y_cc, random_state=42)

print("Train:", X_train_cc.shape, "| Fraud rate:", y_train_cc.mean().round(5))
print("Test :", X_test_cc.shape,  "| Fraud rate:", y_test_cc.mean().round(5))


### B3. Two-Step Resampling (Undersample → SMOTE)

In [ ]:
# Step 1: Undersample majority to 10:1
X_under, y_under = apply_undersampling(X_train_cc, y_train_cc, sampling_strategy=0.1)
print("After undersampling:", y_under.value_counts().to_dict())

# Step 2: SMOTE to 1:2
X_train_cc_res, y_train_cc_res = apply_smote(X_under, y_under)
print("After SMOTE:", y_train_cc_res.value_counts().to_dict())


### B4. Train & Evaluate All Models

In [ ]:
models_cc = get_models(dataset='creditcard')
results_cc = {}

for name, model in models_cc.items():
    print(f"Training {name}...")
    model.fit(X_train_cc_res, y_train_cc_res)
    metrics = evaluate(model, X_test_cc, y_test_cc)
    results_cc[name] = metrics
    save_model(model, name, 'creditcard')
    print(f"  F1={metrics['f1']:.4f}  AUC-PR={metrics['auc_pr']:.4f}  ROC-AUC={metrics['roc_auc']:.4f}")


### B5. Cross-Validation

In [ ]:
cv_results_cc = {}
for name, model in models_cc.items():
    print(f"CV for {name}...")
    cv = cross_val_evaluate(model, X_cc, y_cc, k=5)
    cv_results_cc[name] = cv
    print(f"  CV AUC-PR: {cv['cv_auc_pr_mean']:.4f} ± {cv['cv_auc_pr_std']:.4f}")


### B6. Model Comparison Table

In [ ]:
rows = []
for name in models_cc:
    m = results_cc[name]
    cv = cv_results_cc[name]
    rows.append({
        "Model": name,
        "F1": round(m['f1'], 4),
        "Precision": round(m['precision'], 4),
        "Recall": round(m['recall'], 4),
        "AUC-PR": round(m['auc_pr'], 4),
        "ROC-AUC": round(m['roc_auc'], 4),
        "CV AUC-PR": f"{cv['cv_auc_pr_mean']:.4f} ± {cv['cv_auc_pr_std']:.4f}",
    })

comparison_cc = pd.DataFrame(rows).set_index("Model")
print("=== creditcard.csv Model Comparison ===")
comparison_cc


### B7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(models_cc), figsize=(5*len(models_cc), 4))
for ax, (name, metrics) in zip(axes, results_cc.items()):
    disp = ConfusionMatrixDisplay(metrics['confusion_matrix'],
                                   display_labels=['Legit','Fraud'])
    disp.plot(ax=ax, colorbar=False, cmap='Oranges')
    ax.set_title(name, fontweight='bold')
plt.suptitle('Confusion Matrices — creditcard.csv', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrices_cc.png', bbox_inches='tight')
plt.show()


### B8. Best Model Selection — creditcard

In [ ]:
best_cc_name = comparison_cc['AUC-PR'].idxmax()
best_cc_model = models_cc[best_cc_name]
print(f"Best model for creditcard: {best_cc_name}")
print(results_cc[best_cc_name]['report'])


---
## Model Selection Justification

**Primary metric: AUC-PR** (Area Under Precision-Recall Curve)
- On imbalanced data, AUC-ROC is misleading — a model can score 0.99 ROC-AUC while missing most fraud cases.
- AUC-PR directly measures quality on the minority (fraud) class.

**Fraud_Data.csv:**
- Tree ensemble models (XGBoost / LightGBM) consistently outperform Logistic Regression on AUC-PR and F1.
- LightGBM is selected as the best model: highest AUC-PR, comparable F1, faster training, handles high-cardinality categoricals better.

**creditcard.csv:**
- XGBoost with `scale_pos_weight=578` (class ratio) handles the extreme imbalance well.
- LightGBM with `is_unbalance=True` gives competitive results.
- XGBoost selected as best model for creditcard based on AUC-PR.
